# Stratified VRU results explorer

Read-only notebook over committed CSVs. No training required.

Use this in interviews to walk condition-specific precision/recall without re-running the pipeline.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('.')
if not (ROOT / 'results/finetuned/stratified.csv').exists():
    ROOT = Path('..')

baseline = pd.read_csv(ROOT / 'results/baseline/stratified.csv')
finetuned = pd.read_csv(ROOT / 'results/finetuned/stratified.csv')
summary_b = pd.read_csv(ROOT / 'results/baseline/summary.csv')
summary_f = pd.read_csv(ROOT / 'results/finetuned/summary.csv')
print('loaded', len(baseline), 'baseline rows,', len(finetuned), 'finetuned rows')
summary_f

## Class-level baseline vs fine-tuned

In [ ]:
merged = summary_b.merge(summary_f, on=['class_id','class_name'], suffixes=('_b','_f'))
merged[['class_name','precision_b','recall_b','precision_f','recall_f']]

## Hard-condition recall deltas

In [ ]:
def recall(df, stratum, value, class_name='all'):
    row = df[(df.stratum==stratum)&(df.value==value)&(df.class_name==class_name)]
    return float(row.recall.iloc[0]) if len(row) else float('nan')

rows = []
for label, stratum, value, cls in [
    ('Night all','timeofday','night','all'),
    ('Night person','timeofday','night','person'),
    ('Small','size_bucket','small','all'),
    ('Medium','size_bucket','medium','all'),
    ('City street','scene','city street','all'),
]:
    b = recall(baseline, stratum, value, cls)
    f = recall(finetuned, stratum, value, cls)
    rows.append({'condition': label, 'baseline': b, 'finetuned': f, 'delta': f-b})
hard = pd.DataFrame(rows)
ax = hard.set_index('condition')[['baseline','finetuned']].plot(kind='barh', figsize=(8,4))
ax.set_xlabel('Recall')
ax.set_title('Hard-condition recall')
plt.tight_layout()
hard

## Takeaway

Fine-tuning improves aggregate and some hard strata, but **small-object** and **night-person** recall remain the product risk. Lead with that residual gap in interviews.